In [1]:
import pandas as pd
import json
from collections import defaultdict

In [6]:
# --- Step 1: Load predictions ---
pred_path = "output/results_2.txt"  # your text file
pred_df = pd.read_csv(pred_path, sep=" ", header=None, names=["query", "candidate", "system"])

# normalize (ensure no ".txt" and zero-padded 6-digit IDs)
def normalize_id(x):
    x = str(x).replace(".txt", "")
    return x.zfill(6)

pred_df["query"] = pred_df["query"].apply(normalize_id)
pred_df["candidate"] = pred_df["candidate"].apply(normalize_id)


In [7]:

# --- Step 2: Load ground truth labels ---
with open("./data/task1_test_labels_2025.json", "r") as f:  # replace with your actual test labels JSON path
    gt = json.load(f)

# normalize ground truth
gt = {normalize_id(k): [normalize_id(v) for v in vals] for k, vals in gt.items()}

# --- Step 3: Group predictions by query ---
pred_dict = defaultdict(list)
for _, row in pred_df.iterrows():
    pred_dict[row["query"]].append(row["candidate"])


In [8]:
# --- Step 4: Compute metrics ---
results = []
TP_total, FP_total, FN_total = 0, 0, 0

for query, true_targets in gt.items():
    preds = pred_dict.get(query, [])

    true_set = set(true_targets)
    pred_set = set(preds)

    TP = len(true_set & pred_set)
    FP = len(pred_set - true_set)
    FN = len(true_set - pred_set)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        "query": query,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": TP,
        "FP": FP,
        "FN": FN
    })

    # for micro-averaging
    TP_total += TP
    FP_total += FP
    FN_total += FN


In [9]:


# --- Step 5: Overall (micro) metrics ---
micro_precision = TP_total / (TP_total + FP_total) if (TP_total + FP_total) > 0 else 0
micro_recall = TP_total / (TP_total + FN_total) if (TP_total + FN_total) > 0 else 0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0

metrics_df = pd.DataFrame(results)

print("🔹 Per-query metrics:")
print(metrics_df.head())

print("\n🔸 Overall micro-averaged metrics:")
print(f"Precision: {micro_precision:.20f}")
print(f"Recall:    {micro_recall:.20f}")
print(f"F1-score:  {micro_f1:.20f}")


🔹 Per-query metrics:
    query  precision    recall        f1  TP  FP  FN
0  078507        1.0  0.666667  0.800000   2   0   1
1  023478        0.5  1.000000  0.666667   1   1   0
2  067520        0.0  0.000000  0.000000   0   2   2
3  093289        0.0  0.000000  0.000000   0   1   4
4  090583        0.0  0.000000  0.000000   0   3   2

🔸 Overall micro-averaged metrics:
Precision: 0.39610761305094449591
Recall:    0.39340534394542353569
F1-score:  0.39475185396463202681
